<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 02. Target Encoding y Suavizado: Confiar con Cabeza Fría
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Programación para Ciencia de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 06
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Data%20Science%20programming/06%20-%20Feature%20Engineering/Para%20Dummies/02_Target_Encoding_y_Suavizado_Feature_Engineering_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **"para no ingenieros"** del módulo 02 de Feature Engineering. En el cuaderno anterior de esta serie vimos un adelanto peligroso: reemplazar una categoría por el promedio del precio. Aquí entendemos **por qué** es peligroso y aprendemos a arreglarlo con una técnica llamada **suavizado (smoothing)**.

Al terminar podrás explicar, con tus propias palabras:
1. Los dos problemas del Target Encoding "a lo simple": categorías desconocidas y categorías raras.
2. La idea de mezclar el promedio de la categoría con el promedio global.
3. Cómo funciona la fórmula del "estimador $m$" (m-estimate).
4. Cómo usar la clase `TargetEncoder` de scikit-learn, que hace todo esto de forma automática.

---
## 1. El problema de confiar ciegamente en pocas reseñas ⭐

Piensa en una app de restaurantes. Un restaurante con **200 reseñas** y 4.5 estrellas de promedio te da confianza: es un promedio calculado con muchos datos. Pero un restaurante con **una sola reseña** de 5 estrellas... ¿de verdad es mejor que el de 200 reseñas? Probablemente no lo sabemos todavía — esa única reseña pudo haber sido escrita por el dueño, o por un cliente excepcionalmente feliz un día excepcionalmente bueno.

Esto es exactamente lo que le pasaba a nuestro "adelanto" del cuaderno anterior: calculamos el precio promedio por `tipo` de carro sin fijarnos en **cuántos** carros de ese tipo teníamos. Si un tipo aparecía una sola vez, su "promedio" era, literalmente, el precio de ese único carro.

---
## 2. Dos problemas concretos ⚠️

1. **Categorías desconocidas:** si en el futuro llega una categoría que nunca viste al entrenar (ej. una marca de carro nueva), no tienes ningún promedio calculado para ella — el resultado sería un vacío (`NaN`).
2. **Categorías raras:** si una categoría aparece muy pocas veces, su promedio no es confiable — es como el restaurante con una sola reseña.

Vamos a verlo con datos: un dataset de carros donde una marca (`kia`) aparece **una sola vez**.

In [ ]:
import pandas as pd
import numpy as np

carros = pd.DataFrame({
    "marca": ["toyota", "toyota", "toyota", "ford", "ford", "mazda", "toyota", "ford", "kia"],
    "precio_millones": [70, 75, 68, 60, 58, 65, 72, 62, 90]
})

# Target Encoding "a lo simple": precio promedio por marca
carros["marca_precio_promedio"] = carros.groupby("marca")["precio_millones"].transform("mean")

carros

### 🤔 ¿Qué acaba de pasar?

- Empezamos importando `pandas` y `numpy`, las herramientas base que usaremos en todo el cuaderno.
- `kia` aparece **una sola vez**, con precio 90. Su "promedio por marca" es, entonces, exactamente 90 — no es un promedio real, es el valor de un solo carro disfrazado de promedio.
- Si ese carro `kia` era un caso raro (muy caro o muy barato por alguna razón especial), le estamos enseñando al modelo una regla que probablemente no se sostenga con más datos.
- `toyota`, en cambio, aparece 4 veces, así que su promedio (71.25) sí es razonablemente confiable.

Necesitamos que las marcas con **pocos** datos confíen menos en su propio promedio, y más en el **promedio general** de todos los carros.

---
## 3. La solución: mezclar el promedio de la categoría con el promedio global 💡

La idea, en una frase: **cuantos menos datos tenga una categoría, más debe "jalar" hacia el promedio general**; cuantos más datos tenga, más puede confiar en su propio promedio.

En pseudocódigo:

$$\text{valor codificado} = \text{peso} \times \text{promedio de la categoría} + (1 - \text{peso}) \times \text{promedio general}$$

Donde el `peso` es un número entre 0 y 1 que depende de cuántas veces aparece la categoría. Una forma muy usada de calcular ese `peso` es el llamado **estimador $m$ (m-estimate)**:

$$\text{peso} = \frac{n}{n + m}$$

- $n$: cuántas veces aparece esa categoría en los datos.
- $m$: qué tanto "castigamos" a las categorías con pocos datos (tú lo eliges; valores más grandes exigen más datos antes de confiar en el promedio de la categoría).

In [ ]:
# Paso 1: promedio general de todos los carros
promedio_general = carros["precio_millones"].mean()

# Paso 2: cuantas veces aparece cada marca, y su promedio individual
n = carros.groupby("marca")["precio_millones"].transform("count")
promedio_categoria = carros.groupby("marca")["precio_millones"].transform("mean")

# Paso 3: aplicamos el estimador m con m = 2.0
m = 2.0
peso = n / (n + m)
carros["marca_suavizada"] = peso * promedio_categoria + (1 - peso) * promedio_general

carros[["marca", "precio_millones", "marca_precio_promedio", "marca_suavizada"]]

### 🤔 ¿Qué acaba de pasar?

- `promedio_general` es el precio promedio de **todos** los carros, sin importar la marca.
- `n` cuenta cuántas veces aparece cada marca; `promedio_categoria` es el promedio simple que ya conocíamos.
- Con $m = 2.0$, `kia` (que aparece $n=1$ vez) obtiene un peso de $1 / (1+2) = 0.33$: su valor final es una mezcla de solo 33% su propio precio y 67% el promedio general — mucho más prudente que confiar 100% en un solo dato.
- `toyota` (que aparece $n=4$ veces) obtiene un peso de $4 / (4+2) = 0.67$: confía bastante más en su propio promedio, porque tiene más evidencia detrás.

¡Esto es exactamente la idea del restaurante con muchas reseñas frente al de una sola!

---
## 4. ¿Cómo elegir el valor de $m$? 📈

$m$ controla qué tan exigentes somos antes de confiar en el promedio de una categoría:

- Un $m$ **pequeño** confía rápido en el promedio de la categoría, incluso con pocos datos.
- Un $m$ **grande** exige muchos más datos antes de alejarse del promedio general.

Grafiquemos el `peso` resultante para distintos valores de $m$, a medida que aumenta la cantidad de datos ($n$) que tiene una categoría:

In [ ]:
import matplotlib.pyplot as plt

conteos = np.arange(0, 11)
valores_m = [0.5, 1.0, 2.0, 4.0]

plt.figure(figsize=(7, 4.5))
for m_val in valores_m:
    pesos = conteos / (conteos + m_val)
    plt.plot(conteos, pesos, marker="o", label=f"m = {m_val}")

plt.title("¿Cuánto confiamos en el promedio de una categoria?")
plt.xlabel("Cantidad de datos en la categoria (n)")
plt.ylabel("Peso otorgado al promedio de la categoria")
plt.legend()
plt.tight_layout()
plt.show()

### 🤔 ¿Qué acaba de pasar?

- Cada línea del gráfico representa un valor distinto de $m$.
- Todas las líneas empiezan en `peso = 0` cuando `n = 0` (sin datos, confiamos 100% en el promedio general) y suben hacia `peso = 1` a medida que hay más datos (más confianza en el promedio propio de la categoría).
- Con $m$ más grande (línea más "perezosa"), se necesitan más datos para alcanzar el mismo nivel de confianza.
- No existe un $m$ "correcto" universal: si sospechas que tus categorías son muy ruidosas (varían mucho internamente), usa un $m$ más grande; si son bastante homogéneas, un $m$ pequeño basta.

---
## 5. La versión automática: `TargetEncoder` de scikit-learn 🤖

Calcular el suavizado a mano (como hicimos arriba) es excelente para entender la idea, pero en la práctica scikit-learn ya trae una clase que lo hace por nosotros: `TargetEncoder`. Con `smooth="auto"` incluso elige automáticamente un buen valor equivalente a nuestro $m$.

In [ ]:
from sklearn.preprocessing import TargetEncoder

codificador_objetivo = TargetEncoder(target_type="continuous", smooth="auto", random_state=0)

marca_columna = carros[["marca"]]
precio_objetivo = carros["precio_millones"]

carros["marca_target_encoder"] = codificador_objetivo.fit_transform(marca_columna, precio_objetivo)

carros[["marca", "precio_millones", "marca_suavizada", "marca_target_encoder"]]

### 🤔 ¿Qué acaba de pasar?

- `TargetEncoder(target_type="continuous", smooth="auto")` le dice a scikit-learn: "el objetivo (`precio_millones`) es un número continuo, y elige tú el mejor nivel de suavizado".
- `fit_transform(marca_columna, precio_objetivo)` aprende, en un solo paso, tanto los promedios por marca como el nivel de suavizado adecuado, y devuelve la columna ya codificada.
- Notarás que `marca_target_encoder` es parecido (aunque no idéntico) a nuestro `marca_suavizada` calculado a mano — la diferencia es que scikit-learn usa una técnica interna un poco más sofisticada para elegir el suavizado automáticamente, y además queda listo para usarse con datos nuevos (`transform`) sin recalcular nada.

---
## 6. ¿Cuándo usar Target Encoding con suavizado? 🎯

Esta técnica brilla especialmente en dos casos:

1. **Muchísimas categorías (alta cardinalidad):** columnas como código postal, marca de producto o ciudad, con cientos o miles de valores distintos, donde One-Hot Encoding crearía demasiadas columnas.
2. **Sospechas fundamentadas:** cuando por experiencia sabes que una categoría debería ser informativa, aunque a simple vista no lo parezca.

---
## 7. Resumen relámpago ⚡

| Idea | En una frase |
|---|---|
| Problema del Target Encoding simple | Categorías raras (pocos datos) generan promedios poco confiables; categorías nuevas no tienen promedio. |
| Solución | Mezclar el promedio de la categoría con el promedio general, según cuántos datos tenga. |
| Estimador $m$ | `peso = n / (n + m)`: más datos → más peso al promedio propio de la categoría. |
| Elegir $m$ | $m$ grande = más prudente (exige más datos); $m$ pequeño = confía más rápido. |
| `TargetEncoder` de scikit-learn | Calcula el suavizado automáticamente con `smooth="auto"` y queda listo para datos nuevos. |

➡️ **Siguiente paso:** en el cuaderno **03 - Creación de Características (Para Dummies)** aprenderás a construir nuevas columnas a partir de operaciones matemáticas, fechas y agrupaciones — el siguiente gran truco del Feature Engineering.

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Programación para Ciencia de Datos (Edición Para No Ingenieros)</i>
  </p>
</div>